In [1]:
## Setup
!pip install -q \
    "numpy<2.0" \
    "openai-whisper>=20231117" \
    "edge-tts>=6.1.9" \
    "langchain>=0.3" \
    "langchain-core>=0.3" \
    "langchain-google-genai>=2.0" \
    "google-ai-generativelanguage>=0.6.10" \
    "gradio>=4.40" \
    "nest-asyncio>=1.6" \
    "langchain-groq"

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 3.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 803.2/803.2 kB 17.9 MB/s eta 0:00:0000:01
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 93.5 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.4/69.4 kB 8.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 17.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 197.7/197.7 MB 5.7 MB/s eta 0:00:00:00:0100:01
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
opencv-python 4.13.0.92 requires numpy>=2; python_version >= "3.9", but you have numpy 1.26.4 which is incompatible.
tobler 0.14.0 requires numpy>=2.0, but you have numpy 1.26.4 which 

In [1]:
## set Groq key
import os
try:
    from google.colab import userdata
    os.environ["GROQ_API_KEY"] = userdata.get("GROQ_API_KEY")
    print("API key loaded from Colab secrets ✓")
except Exception:
    if "GROQ_API_KEY" not in os.environ:
        from getpass import getpass
        os.environ["GROQ_API_KEY"] = getpass("Paste your GROQ_API_KEY API key: ")
    print("API key set ✓")

API key set ✓


In [5]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
## Get a sample audio file
import urllib.request
from pathlib import Path
from google.colab import files

SAMPLE_URL = "https://github.com/openai/whisper/raw/main/tests/jfk.flac"
folder = Path("/content/drive/MyDrive/voiceAgent")
folder.mkdir(parents=True, exist_ok=True)

sample_path = folder / "sample.flac"

if not sample_path.exists():
    urllib.request.urlretrieve(SAMPLE_URL, sample_path)

print(f"Sample audio saved to {sample_path}")
print(f"Size: {sample_path.stat().st_size / 1024:.1f} KB")

# Play it inline so we can hear what we are working with
from IPython.display import Audio, display
display(Audio(str(sample_path)))

FileNotFoundError: [Errno 2] No such file or directory: '/content/drive/MyDrive/voiceAgent/sample.flac'

## Stage 1: Speech to text using Whisper

Whisper is OpenAI open source speech recognition model. Whisper does more than transcribe. It detects the language automatically, handles punctuation, and gives you per segment timestamps.

## Load the Whisper Model

In [3]:
import whisper

# "base" is 74MB, takes about 30 seconds to download on first run
whisper_model = whisper.load_model("base")
print(f"Whisper base loaded. Multilingual: {whisper_model.is_multilingual}")

100%|███████████████████████████████████████| 139M/139M [00:01<00:00, 81.5MiB/s]


Whisper base loaded. Multilingual: True
